# 05 Optimization (Phase 6)

This notebook follows `energy_optimizer_colab_updated` Phase 6:
- Define tariff schedule
- Run LP over 48-hour windows
- Validate feasibility and savings on 5 windows
- Save/deploy `lp_scheduler.py`

In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import json
from scipy.optimize import linprog

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/energy-optimizer'
print(f"Storage Bound: {BASE}")

Mounted at /content/drive
Storage Bound: /content/drive/MyDrive/energy-optimizer


In [ ]:
# Bootstrap recent_history.csv from Drive raw data
RAW_PATH = os.path.join(BASE, 'data/raw/household_power_consumption.csv')
OUT_PATH = os.path.join(BASE, 'data/processed/recent_history.csv')

df = pd.read_csv(RAW_PATH, sep=';', na_values=['?'], dtype={'Global_active_power': 'float64'})
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)

df_hourly = df[['Global_active_power']].resample('1H').mean()
df_hourly.rename(columns={'Global_active_power': 'consumption'}, inplace=True)
df_hourly['consumption'] = df_hourly['consumption'].ffill(limit=3)
df_hourly['consumption'] = df_hourly['consumption'].interpolate(method='linear', limit=24)
df_hourly.dropna(inplace=True)

recent_history = df_hourly.tail(200).copy()
recent_history.index.name = 'Datetime'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
recent_history.to_csv(OUT_PATH)

print(f"Saved recent history: {OUT_PATH}")
print(f"Rows: {len(recent_history)} | Range: {recent_history.index.min()} -> {recent_history.index.max()}")

/tmp/ipykernel_2220/243133424.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df[['Global_active_power']].resample('1H').mean()


Saved recent history: /content/drive/MyDrive/energy-optimizer/data/processed/recent_history.csv
Rows: 200 | Range: 2010-11-18 14:00:00 -> 2010-11-26 21:00:00


## Tariff schedule

In [ ]:
TARIFF = {
    "peak_hours": list(range(16, 21)),
    "mid_peak_hours": list(range(7, 16)),
    "off_peak_hours": list(range(0, 7)) + list(range(21, 24)),
    "peak_rate": 9.0,
    "mid_peak_rate": 6.5,
    "off_peak_rate": 3.5,
}

def build_tariff_vector(horizon: int, tariff: dict) -> np.ndarray:
    daily = np.full(24, tariff["mid_peak_rate"], dtype=float)
    for h in tariff["peak_hours"]:
        daily[h] = tariff["peak_rate"]
    for h in tariff["off_peak_hours"]:
        daily[h] = tariff["off_peak_rate"]
    return np.array([daily[i % 24] for i in range(horizon)], dtype=float)

## LP formulation

In [ ]:
def optimize_window(forecast, capacity, flexibility_fraction=0.20):
    x = np.asarray(forecast, dtype=float)
    rates = build_tariff_vector(len(x), TARIFF)

    total_demand = float(x.sum())
    A_eq = np.ones((1, len(x)))
    b_eq = np.array([total_demand])

    bounds = []
    for i in range(len(x)):
        flex_limit = x[i] * flexibility_fraction
        lower = max(0.0, x[i] - flex_limit)
        upper = min(capacity, x[i] + flex_limit)
        bounds.append((lower, upper))

    res = linprog(c=rates, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    original_cost = float(np.dot(rates, x))
    if res.status != 0:
        return {"status": int(res.status), "x": x, "original_cost": original_cost, "optimized_cost": original_cost}

    optimized = np.asarray(res.x, dtype=float)
    optimized_cost = float(np.dot(rates, optimized))
    return {
        "status": int(res.status),
        "x": optimized,
        "original_cost": original_cost,
        "optimized_cost": optimized_cost,
        "savings": original_cost - optimized_cost,
    }

## Test on 5 real 48-hour windows

In [ ]:
df = pd.read_csv(os.path.join(BASE, 'data/processed/recent_history.csv'), parse_dates=["Datetime"])
capacity_multiplier = 1.25
capacity = float(df["consumption"].max() * capacity_multiplier)
positions = [0, 30, 60, 90, 120]
rows = []

for pos in positions:
    w = df.iloc[pos:pos+48]
    if len(w) < 48:
        continue
    out = optimize_window(w["consumption"].tolist(), capacity=capacity, flexibility_fraction=0.20)
    x = np.asarray(out["x"], dtype=float)
    total_demand = float(w["consumption"].sum())
    rows.append({
        "window_start": w["Datetime"].iloc[0],
        "window_end": w["Datetime"].iloc[-1],
        "status": out["status"],
        "optimized_cost_leq_original": out["optimized_cost"] <= out["original_cost"] + 1e-9,
        "demand_conserved": abs(x.sum() - total_demand) <= 1e-6,
        "all_non_negative": bool((x >= -1e-9).all()),
        "within_capacity": bool((x <= capacity + 1e-9).all()),
        "original_cost": out["original_cost"],
        "optimized_cost": out["optimized_cost"],
        "savings": out["original_cost"] - out["optimized_cost"],
        "savings_pct": (out["original_cost"] - out["optimized_cost"]) / out["original_cost"] * 100 if out["original_cost"] > 0 else 0.0,
    })

results = pd.DataFrame(rows)
results

,window_start,window_end,status,optimized_cost_leq_original,demand_conserved,all_non_negative,within_capacity,original_cost,optimized_cost,savings,savings_pct
0,2010-11-18 14:00:00,2010-11-20 13:00:00,0,True,True,True,True,312.059150,288.091693,23.967457,7.680421
1,2010-11-19 20:00:00,2010-11-21 19:00:00,0,True,True,True,True,284.608933,265.614500,18.994433,6.673871
2,2010-11-21 02:00:00,2010-11-23 01:00:00,0,True,True,True,True,311.597700,295.406507,16.191193,5.196185
3,2010-11-22 08:00:00,2010-11-24 07:00:00,0,True,True,True,True,312.256867,292.875700,19.381167,6.206802
4,2010-11-23 14:00:00,2010-11-25 13:00:00,0,True,True,True,True,303.788550,281.104260,22.684290,7.467131


## Save `lp_scheduler.py` to Drive

In [ ]:
scheduler_py = '''import numpy as np
from scipy.optimize import linprog

TARIFF = {
    "peak_hours": list(range(16, 21)),
    "mid_peak_hours": list(range(7, 16)),
    "off_peak_hours": list(range(0, 7)) + list(range(21, 24)),
    "peak_rate": 9.0,
    "mid_peak_rate": 6.5,
    "off_peak_rate": 3.5,
}


def build_tariff_vector(horizon: int, tariff: dict = TARIFF) -> np.ndarray:
    daily = np.full(24, tariff["mid_peak_rate"], dtype=float)
    for h in tariff["peak_hours"]:
        daily[h] = tariff["peak_rate"]
    for h in tariff["off_peak_hours"]:
        daily[h] = tariff["off_peak_rate"]
    return np.array([daily[i % 24] for i in range(horizon)], dtype=float)


def optimize_window(forecast, capacity, flexibility_fraction=0.20):
    x = np.asarray(forecast, dtype=float)
    rates = build_tariff_vector(len(x), TARIFF)

    total_demand = float(x.sum())
    A_eq = np.ones((1, len(x)))
    b_eq = np.array([total_demand])

    bounds = []
    for i in range(len(x)):
        flex_limit = x[i] * flexibility_fraction
        lower = max(0.0, x[i] - flex_limit)
        upper = min(capacity, x[i] + flex_limit)
        if lower > upper:
            lower = upper
        bounds.append((lower, upper))

    res = linprog(c=rates, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    original_cost = float(np.dot(rates, x))
    if res.status != 0:
        return {
            "status": int(res.status),
            "x": x,
            "original_cost": original_cost,
            "optimized_cost": original_cost,
            "savings": 0.0,
            "savings_pct": 0.0,
        }

    optimized = np.asarray(res.x, dtype=float)
    optimized_cost = float(np.dot(rates, optimized))
    savings = original_cost - optimized_cost
    return {
        "status": int(res.status),
        "x": optimized,
        "original_cost": original_cost,
        "optimized_cost": optimized_cost,
        "savings": savings,
        "savings_pct": (savings / original_cost * 100.0) if original_cost > 0 else 0.0,
    }
'''

scheduler_path = os.path.join(BASE, 'src/optimization/lp_scheduler.py')
os.makedirs(os.path.dirname(scheduler_path), exist_ok=True)
with open(scheduler_path, 'w') as f:
    f.write(scheduler_py)

print(f"Saved: {scheduler_path}")

Saved: /content/drive/MyDrive/energy-optimizer/src/optimization/lp_scheduler.py


## Update metadata on Drive

In [ ]:
metadata_path = os.path.join(BASE, 'artifacts/metadata.json')
with open(metadata_path) as f:
    metadata = json.load(f)

metadata['lp_config'] = {
    'flexibility_fraction': 0.20,
    'capacity_multiplier': 1.25
}
metadata['tariff'] = {
    'peak': 9.0,
    'mid_peak': 6.5,
    'off_peak': 3.5
}
metadata['lp_scheduler_path'] = 'src/optimization/lp_scheduler.py'

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print('Updated metadata.json')
print(json.dumps(metadata, indent=2))

Updated metadata.json
{
  "version_tag": "Production-v1",
  "algorithm": "LightGBM",
  "model_filename": "forecast_winner.pkl",
  "q05_filename": "xgb_q05.pkl",
  "q95_filename": "xgb_q95.pkl",
  "scaler_filename": "main_scaler.pkl",
  "feature_columns": "feature_columns.json",
  "metrics": {
    "test_rmse": 0.5035,
    "test_mae": 0.3483
  },
  "anomaly_model_filename": "isolation_forest.pkl",
  "anomaly_features": "anomaly_feature_columns.json",
  "anomaly_metrics": {
    "precision": 0.846,
    "recall": 0.733,
    "f1": 0.786
  },
  "anomaly_config": {
    "gate_logic": "AND",
    "contamination": 0.01,
    "sigma_threshold": 3.5,
    "abs_threshold_kwh": 1.5,
    "ratio_threshold": 2.0
  },
  "lp_config": {
    "flexibility_fraction": 0.2,
    "capacity_multiplier": 1.25
  },
  "tariff": {
    "peak": 9.0,
    "mid_peak": 6.5,
    "off_peak": 3.5
  },
  "lp_scheduler_path": "src/optimization/lp_scheduler.py"
}


## Phase 6 checklist

- [ ] LP formulation documented
- [ ] Tariff schedule defined
- [ ] 5 windows tested with all checks passing
- [ ] `lp_scheduler.py` saved and copied to local `src/optimization/`